In [55]:
import json
import pandas as pd
import networkx as nx
from dateutil.parser import parse
import networkx as nx
from collections import defaultdict
import os

In [56]:
with open('dataset/services.json', 'r',encoding="utf8") as file:
    services_data = json.load(file)

In [57]:
print("Các thông tin có trong file:", services_data.keys())
print(f"Số lượng services: {len(services_data['services'])}")
print(f"Số lượng stores: {len(services_data['stores'])}")
print(f"Số lượng edges: {len(services_data['edges'])}")
print(f'Các services: {list(services_data['services'])}')
print(f'Các stores: {list(services_data['stores'])}')
print(f'Các edges: {list(services_data['edges'])}')
print()
for _ in services_data:
    print(services_data)

Các thông tin có trong file: dict_keys(['_meta', 'services', 'stores', 'edges', 'topology_notes'])
Số lượng services: 10
Số lượng stores: 4
Số lượng edges: 17
Các services: [{'name': 'edge-lb', 'tier': 'edge', 'team': 'platform', 'owner_pager': 'platform-oncall', 'criticality': 'high', 'nab_stream': 'realKnownCause/ec2_request_latency_system_failure.csv'}, {'name': 'auth-svc', 'tier': 'api', 'team': 'identity', 'owner_pager': 'identity-oncall', 'criticality': 'high', 'nab_stream': 'realKnownCause/ambient_temperature_system_failure.csv'}, {'name': 'checkout-svc', 'tier': 'api', 'team': 'checkout', 'owner_pager': 'checkout-oncall', 'criticality': 'high', 'nab_stream': 'realAWSCloudwatch/ec2_cpu_utilization_5f5533.csv'}, {'name': 'payment-svc', 'tier': 'api', 'team': 'payments', 'owner_pager': 'payments-oncall', 'criticality': 'critical', 'nab_stream': 'realAWSCloudwatch/ec2_cpu_utilization_77c1ca.csv'}, {'name': 'cart-svc', 'tier': 'api', 'team': 'checkout', 'owner_pager': 'checkout-onca

#### File này ở định dạng .jsonl (JSON Lines) — tức là mỗi dòng là một đối tượng JSON độc lập đại diện cho 1 alert. ==> dùng pandas

In [58]:
alert_data = []

with open('dataset/alerts_sample.jsonl', 'r',encoding="utf8" ) as alert_file:
    for line in alert_file:
        line = line.strip()

        if line:
            try:
                alert_data.append(json.loads(line))
            except json.JSONDecodeError:
                continue

alert_df = pd.DataFrame(alert_data)
print(f' Đọc được {len(alert_df)} dòng!')
print(alert_df.info)
print()
print(f'\n{alert_df.head()}')

 Đọc được 20 dòng!
<bound method DataFrame.info of         id                    ts           service  \
0   a-0001  2026-06-12T09:42:01Z       payment-svc   
1   a-0002  2026-06-12T09:42:18Z       payment-svc   
2   a-0003  2026-06-12T09:42:22Z       payment-svc   
3   a-0004  2026-06-12T09:42:30Z       payment-svc   
4   a-0005  2026-06-12T09:42:45Z      checkout-svc   
5   a-0006  2026-06-12T09:43:01Z      checkout-svc   
6   a-0007  2026-06-12T09:43:15Z           edge-lb   
7   a-0008  2026-06-12T09:43:18Z       payment-svc   
8   a-0009  2026-06-12T09:43:32Z          cart-svc   
9   a-0010  2026-06-12T09:43:50Z  notification-svc   
10  a-0011  2026-06-12T09:44:02Z       payment-svc   
11  a-0012  2026-06-12T09:44:30Z      checkout-svc   
12  a-0013  2026-06-12T09:45:10Z   recommender-svc   
13  a-0014  2026-06-12T09:45:42Z           edge-lb   
14  a-0015  2026-06-12T09:46:01Z       payment-svc   
15  a-0016  2026-06-12T09:46:50Z        search-svc   
16  a-0017  2026-06-12T09:47:12

In [59]:
def fingerprint(alert: dict) -> str:
    return f"{alert['service']}|{alert['metric']}|{alert['severity']}"

In [60]:
class Deduper:
    def __init__(self):
        self.store: dict[str, dict] = {}

    def push(self, alert: dict) -> str:
        fp = fingerprint(alert)
        if fp not in self.store:
            self.store[fp] = {
                'cluster_id': fp,
                'count': 1,
                'first_seen': alert['ts'],
                'last_seen': alert['ts'],
                'alerts': [alert['id']],
            }
        else:
            c = self.store[fp]
            c['count'] += 1
            c['last_seen'] = alert['ts']
            c['alerts'].append(alert['id'])
        return fp


In [61]:
raw_alert = alert_df.to_dict('records')

deduper = Deduper()

for data in raw_alert:
    deduper.push(data)

print(f"Số lượng alert thô ban đầu: {len(raw_alert)}")
print(f"Số lượng vân tay (fingerprint) duy nhất sau Layer 1: {len(deduper.store)}")

Số lượng alert thô ban đầu: 20
Số lượng vân tay (fingerprint) duy nhất sau Layer 1: 17


In [62]:
def session_groups(alerts: list[dict], gap_sec: int = 120) -> list[list[dict]]:
    """Mỗi group là 1 'session'. Session ngắt khi gap > gap_sec giây."""
    if not alerts:
        return []
    sorted_alerts = sorted(alerts, key=lambda a: a['ts'])
    groups = [[sorted_alerts[0]]]
    for alert in sorted_alerts[1:]:
        last_ts = parse(groups[-1][-1]['ts'])
        if (parse(alert['ts']) - last_ts).total_seconds() <= gap_sec:
            groups[-1].append(alert)
        else:
            groups.append([alert])
    return groups


In [63]:
ses_window = session_groups(alerts=raw_alert, gap_sec=45)

print(f"Từ {len(raw_alert)} alerts ban đầu...")
print(f"Hệ thống chia thành {len(ses_window)} khoảng thời gian (Sessions) khác nhau.")

for i, session in enumerate(ses_window):
    print(f" - Session {i + 1}: chứa {len(session)} alerts")

Từ 20 alerts ban đầu...
Hệ thống chia thành 3 khoảng thời gian (Sessions) khác nhau.
 - Session 1: chứa 15 alerts
 - Session 2: chứa 2 alerts
 - Session 3: chứa 3 alerts


In [64]:
def build_graph(services_data: dict) -> nx.DiGraph:
    g = nx.DiGraph()
    for svc in services_data['services']:
        g.add_node(svc['name'], **{k: v for k, v in svc.items() if k != 'name'})
        
    for store in services_data.get('stores', []): # Sửa ở đây
        g.add_node(store['name'], **{k: v for k, v in store.items() if k != 'name'})
        
    for edge in services_data.get('edges', []): # Và sửa ở đây
        g.add_edge(edge['from'], edge['to'], type=edge['type'])
        
    return g


In [65]:
def topology_group(alerts, graph, max_hop=2):
    """Gom alert có service cách nhau ≤ max_hop trên graph."""
    undirected = graph.to_undirected()
    by_service = defaultdict(list)
    for a in alerts:
        by_service[a['service']].append(a)

    services = list(by_service.keys())
    parent = {s: s for s in services}
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for i, s1 in enumerate(services):
        for s2 in services[i+1:]:
            try:
                if nx.shortest_path_length(undirected, s1, s2) <= max_hop:
                    parent[find(s1)] = find(s2)
            except nx.NetworkXNoPath:
                pass

    groups = defaultdict(list)
    for s in services:
        groups[find(s)].extend(by_service[s])
    return list(groups.values())

In [66]:

first_session = ses_window[0]
print(f"Session 1 ban đầu có {len(first_session)} alerts, gồm các service:")
print(set([a['service'] for a in first_session]))

topology = topology_group(first_session, graph=build_graph(services_data),max_hop=2)

print(f"Layer 3 đã xé Session 1 ra thành {len(topology)} Cluster nhỏ hơn:")
for i, cluster in enumerate(topology):
    print(f" - Cluster {i+1}: {set([a['service'] for a in cluster])}")

Session 1 ban đầu có 15 alerts, gồm các service:
{'notification-svc', 'edge-lb', 'cart-svc', 'recommender-svc', 'checkout-svc', 'payment-svc'}
Layer 3 đã xé Session 1 ra thành 1 Cluster nhỏ hơn:
 - Cluster 1: {'notification-svc', 'edge-lb', 'cart-svc', 'recommender-svc', 'checkout-svc', 'payment-svc'}


In [70]:
import os

def correlate(alerts: list[dict], graph: nx.DiGraph, gap_sec=45, max_hop=2):
    sessions = session_groups(alerts, gap_sec=gap_sec)
    
    clusters = []
    for s_idx, session_alerts in enumerate(sessions):
        for g_idx, group in enumerate(topology_group(session_alerts, graph, max_hop)):
            
            clusters.append({
                'cluster_id': f'c-{s_idx:03d}-{g_idx:03d}',
                'alert_count': len(group),
                'services': sorted(list({a['service'] for a in group})),
                'time_range': [
                    min(a['ts'] for a in group), 
                    max(a['ts'] for a in group)
                ],
                'max_severity': 'crit' if any(a['severity'] == 'critical' for a in group) 
                                else max(a['severity'] for a in group),
                'alert_ids': [a['id'] for a in group]
            })
    return clusters

final_clusters = correlate(raw_alert, graph=build_graph(services_data), gap_sec=30, max_hop=2)

input_alerts = len(raw_alert)
output_clusters = len(final_clusters)
reduction_ratio = 1 - (output_clusters / input_alerts)

output_data = {
    "input_alerts": input_alerts,
    "output_clusters": output_clusters,
    "reduction_ratio": round(reduction_ratio, 2),
    "clusters": final_clusters
}

# # 3. Lưu ra file JSON
os.makedirs('results', exist_ok=True) # Tạo folder results nếu chưa có
with open('results/cluster_summary.json', 'w') as file:
    json.dump(output_data, file, indent=2)

print(f"Số Alerts ban đầu: {input_alerts}")
print(f"Số Clusters chốt lại: {output_clusters}")
print(f"Tỷ lệ giảm tải (Reduction Ratio): {reduction_ratio * 100:.1f}%")

Số Alerts ban đầu: 20
Số Clusters chốt lại: 5
Tỷ lệ giảm tải (Reduction Ratio): 75.0%
